In [1]:
# !wget https://github.com/3alae-22/llm-zoomcamp-2026-agents/blob/main/ingest.py



In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

79

In [4]:
documents=documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
import instructor
from anthropic import Anthropic

anthropic_client = instructor.from_anthropic(Anthropic())

In [10]:
import json

user_prompt = json.dumps(doc)

In [11]:
messages = [
    {"role": "user", "content": user_prompt}
]

In [12]:
response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        system=data_gen_instructions,
        messages=messages,
        response_model=Questions,
        max_tokens=512
    )

In [13]:
response.questions

['Is it too late to enroll in this course if I just found out about it?',
 'Can I still get a certificate if I start the course now?',
 "What's the deadline for submitting projects to get certified?",
 'Do I need to join at a specific time to receive a certificate?',
 'If I sign up late, will I be able to complete the course and get verified?']

In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    anthropic_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to join this course if I just found out about it?', 'Do I need to enroll right away to get a certificate, or can I join later?', "What's the deadline for submitting projects if I want to earn the certificate?", "Can I still participate in the course if I'm starting late, and will I be able to get certified?", 'How long do you usually accept project submissions for getting a certificate?']


In [16]:
result.questions

['Is it too late to join this course if I just found out about it?',
 'Do I need to enroll right away to get a certificate, or can I join later?',
 "What's the deadline for submitting projects if I want to earn the certificate?",
 "Can I still participate in the course if I'm starting late, and will I be able to get certified?",
 'How long do you usually accept project submissions for getting a certificate?']

In [20]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to join this course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to enroll right away to get a certificate, or can I join later?',
  'document': '74eb249bbf'},
 {'question': "What's the deadline for submitting projects if I want to earn the certificate?",
  'document': '74eb249bbf'},
 {'question': "Can I still participate in the course if I'm starting late, and will I be able to get certified?",
  'document': '74eb249bbf'},
 {'question': 'How long do you usually accept project submissions for getting a certificate?',
  'document': '74eb249bbf'}]

In [21]:
import pandas as pd

In [22]:
pd.DataFrame(records)

,question,document
0,Is it too late to join this course if I just f...,74eb249bbf
1,Do I need to enroll right away to get a certif...,74eb249bbf
2,What's the deadline for submitting projects if...,74eb249bbf
3,Can I still participate in the course if I'm s...,74eb249bbf
4,How long do you usually accept project submiss...,74eb249bbf


In [ ]:
from evaluation_utils import llm_structured_retry

In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        anthropic_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
generate_ground_truth(doc)

([{'question': 'Is it too late to sign up for the course if I just found out about it?',
   'document': '74eb249bbf'},
  {'question': 'Can I join if the course has already started?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to enroll right now to get a certificate later?',
   'document': '74eb249bbf'},
  {'question': "What's the deadline for submitting a project to get certified?",
   'document': '74eb249bbf'},
  {'question': 'If I join late, will I still be able to complete the course and get recognized?',
   'document': '74eb249bbf'}],
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=853, output_tokens=127, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [25]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/79 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

395

In [37]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.10792800000000001

In [38]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [39]:
df_ground_truth.head()

,question,document
0,Is it too late to enroll in the course right now?,74eb249bbf
1,Can I still get a certificate if I join at thi...,74eb249bbf
2,What's the deadline for submitting the final p...,74eb249bbf
3,"If I start the course now, will I be able to c...",74eb249bbf
4,Do I need to do anything special if I'm joinin...,74eb249bbf


In [41]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)